#### Import libraries

In [ ]:
from collections import deque
from datetime import datetime
from fugashi import Tagger
import glob
import IPython
import jaconv
from jiwer import process_characters
import json
import librosa
import math
from matplotlib import pyplot as plt
import numpy as np
import os
import pandas as pd
import pydomino
import re
from scipy import stats as ss
from scipy.cluster.hierarchy import linkage, fcluster
import shutil
from sklearn.cluster import KMeans
import soundfile as sf
import torch
import transformers
import whisper

#### constant

In [ ]:
os.environ["OMP_NUM_THREDS"] = "1"

In [ ]:
frame_quantizes = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
lab_time_magnification = 10**7
min_aligned_timeframe = 1
max_quantize = 192
min_temp_measure = 1
max_temp_duration = 30
samplingrate = 48000
segmentation_rate = 5
threshold_normalized_rms = 0.2
threshold_normalized_rms_minlimit = 0.01
threshold_segmentation_time = 30

In [ ]:
threshold_whisper_temperature = 0
whisper_language = "ja"

In [ ]:
_katakana_re = re.compile(r"[\u30A0-\u30FF]+")
kana = pd.read_csv("kana.csv", index_col=0, encoding="utf-8")
phone_con = "cl"
kana_con = "っ"
phone_N = "N"
kana_N = "ん"
phone_vowel = list(kana.columns)
kana_vowel = list(kana.iloc[0, :])
kana_sulky = ["ぁ", "ぃ", "ぅ", "ぇ", "ぉ", "ゃ", "ゅ", "ょ"]
long_vowel = "ー"
phone_rest = "pau"

In [ ]:
fmin = 50
fmax = 2100
tuning_frequency = 440
A_lowest = tuning_frequency/(2**4)
rest = "pau"
rest_number = 88
scale_names = {
    0: "A",
    1: "A#",
    2: "B",
    3: "C",
    4: "C#",
    5: "D",
    6: "D#",
    7: "E",
    8: "F",
    9: "F#",
    10: "G",
    11: "G#"
}
num_scale = len(scale_names)

In [ ]:
key_numbers = {
    "C": 0,
    "Am": 0,
    "G": 1,
    "Em": 1,
    "D": 2,
    "Bm": 2,
    "A": 3,
    "F#m": 3,
    "E": 4,
    "C#m": 4,
    "B": 5,
    "G#m": 5,
    "F#": 6,
    "D#m": 6,
    "C#": 7,
    "A#m": 7,
    "F": -1,
    "Dm": -1,
    "Bb": -2,
    "Gm": -2,
    "Eb": -3,
    "Cm": -3,
    "Ab": -4,
    "Fm": -4,
    "Db": -5,
    "Bbm": -5,
    "Gb": -6,
    "Ebm": -6,
    "Cb": -7,
    "Abm": -7
}

In [ ]:
alignmer = pydomino.Aligner("./pydomino/onnx_model/phoneme_transition_model.onnx")
#model = whisper.load_model("large-v3", device="cuda")
model = whisper.load_model("turbo", device="cuda")
#model = whisper.load_model("turbo", device="cpu")
pipe = transformers.pipeline(
    "automatic-speech-recognition",
    model="Parakeet-Inc/furigana_whisper_small_jsut"
)
tagger = Tagger()

In [ ]:
SV2_divisions = 705600000
SV2_pitch_correction = 21

In [ ]:
random_state = 6

#### global variables

In [ ]:
min_y = None
max_y = None
max_rms = None
temp_id = 0

#### music_segmentation

In [ ]:
def plot(y=None, wave=None, rms=None, thre=False, beat_bar=None):
    plt.figure(figsize=(24, 4))
    if y is not None:
        plt.plot(y)
    if rms is not None:
        plt.plot(rms)
        if thre:
            plt.plot(np.full(rms.shape[0], threshold_normalized_rms), color="r")
    if beat_bar is not None:
        plt.plot(beat_bar, alpha=0.3, color="g")
    plt.tight_layout()
    plt.show()
    if wave is not None:
        display(IPython.display.Audio(wave, rate=samplingrate))

In [ ]:
def print_outputlog(text):
    print("output: {}  ({})".format(text, datetime.now().strftime("%Y/%m/%d %H:%M:%S")))

In [ ]:
def print_deletelog(text):
    print("output: {}  ({})".format(text, datetime.now().strftime("%Y/%m/%d %H:%M:%S")))

In [ ]:
def delete_file(file):
    os.remove(file)
    print_deletelog(file)

In [ ]:
def stretch_1darray(short_array, long_array):
    short_length = short_array.shape[0]
    long_length = long_array.shape[0]
    stretch_rate = long_length / short_length

    stretched_array = np.zeros(long_length)
    prev_arg = 0
    for i in range(short_length):
        arg = int(stretch_rate*i)
        value = short_array[i]
        stretched_array[prev_arg:arg] = value
        prev_arg = arg
    stretched_array[prev_arg:] = value
    return stretched_array

In [ ]:
def compute_rms(y, frame_length, hop_length, center=True):
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length, center=center)
    rms = stretch_1darray(rms[0][:frame_length//hop_length], y)
    return rms

In [ ]:
def measurement_rms(meta, y, rms=None, beat_df=None, frame_quantize=4, start="1-1", end=None):
    global max_rms

    y_absoluted = np.abs(y)
    y_normalized = (y_absoluted-min_y) / (max_y-min_y)

    title = meta["title"]
    measure = meta["measure"]
    quantize = meta["quantize"]

    beat = meta["beat"]
    beats_csv_path = "./meta/{}/beats.csv".format(title)
    if os.path.isfile(beats_csv_path):
        beats = pd.read_csv(beats_csv_path)
    else:
        beats = pd.DataFrame([[1, beat]], columns=["measure_number", "beat"])
    nume_beat, denomi_beat = map(int, re.findall(r'[0-9]+', beat))
    beat_part = quantize//denomi_beat
    measure_part = beat_part*nume_beat
    change_beat_count = beats.shape[0]

    bpm = meta["bpm"]
    bpms_csv_path = "./meta/{}/bpms.csv".format(title)
    if os.path.isfile(bpms_csv_path):
        bpms = pd.read_csv(bpms_csv_path)
    else:
        bpms = pd.DataFrame([[1, 1, bpm]], columns=["measure_number", "beat_number", "bpm"])
    change_bpm_count = bpms.shape[0]

    preparation = meta["preparation"]
    preparation_finish = 0
    lingering = meta["lingering"]
    lingering_start = 1

    duration = (60/bpm)*(4/quantize)  #1クオンタイズの時間
    frame_duration = (60/bpm)*(4/frame_quantize)  #rmsを計算する時間幅
    frame_length = int(samplingrate*frame_duration)  #rmsを計算するサンプリングフレーム数
    hop_duration = (60/bpm)*(4/quantize)  #rmsを計算するウィンドウ時間幅（1クオンタイズ分）
    hop_length = int(samplingrate*hop_duration)  #rmsを計算するウィンドウサンプリングフレーム数

    full_duration = librosa.get_duration(y=y, sr=samplingrate)  #音源の長さ
    full_frame = y.shape[0]  #音源のフレーム数
    #print("title: {}, duration: {}s ({}f)".format(title, full_duration, full_frame))

    start_measure, start_beat = map(int, re.findall(r'[0-9]+', start))
    start_frame = 0
    is_start = False
    end_measure, end_beat = map(int, re.findall(r'[0-9]+', "1-1"))
    if end is None:
        end_measure, end_beat = measure, measure_part
    else:
        end_measure, end_beat = map(int, re.findall(r'[0-9]+', end))
    end_frame = 0
    is_end = False
    
    if rms is None:
        rms = np.zeros(full_frame)
    if beat_df is None:
        beat_df = pd.DataFrame([], columns=["measure_number", "beat_number", "time", "frame", "frame_quantize", "section"])
    beat_bar = np.zeros(full_frame)
    current_change_beat_count = 0
    current_change_bpm_count = 0
    preview_offset = 0
    offset = 0
    offset_frame = 0
    part_count = 0
    bpm_part_count = 0
    for i in range(1, measure+1):
        #print("---measure number: {}---".format(i))
        if current_change_beat_count < change_beat_count and beats.iat[current_change_beat_count, 0] == i:
            beat = beats.iat[current_change_beat_count, 1]
            nume_beat, denomi_beat = map(int, re.findall(r'[0-9]+', beat))
            beat_part = quantize//denomi_beat
            measure_part = beat_part*nume_beat
            if end is None:
                end_beat = measure_part
            current_change_beat_count += 1
        if preparation > 0:
            if nume_beat > preparation:
                preparation_finish += beat_part*preparation
                preparation = 0
            else:
                preparation_finish += measure_part
                preparation -= nume_beat
        if lingering > 0:
            if nume_beat > lingering:
                lingering_start += beat_part*(nume_beat - lingering)
                lingering = 0
            else:
                lingering -= nume_beat
        else:
            lingering_start += measure_part
        for j in range(1, measure_part+1):
            offset = duration*bpm_part_count + preview_offset
            offset_frame = int((offset/full_duration)*full_frame)
            beat_bar[offset_frame] = 1
            #print("beat number: {} (t = {}s, frame = {}f)".format(j, offset, offset_frame))
            if current_change_bpm_count < change_bpm_count and bpms.iat[current_change_bpm_count, 0] == i and bpms.iat[current_change_bpm_count, 1] == j:
                bpm = bpms.iat[current_change_bpm_count, 2]
                duration = (60/bpm)*(4/quantize)
                frame_duration = (60/bpm)*(4/frame_quantize)
                frame_length = int(samplingrate*frame_duration)
                hop_duration = (60/bpm)*(4/quantize)
                hop_length = int(samplingrate*hop_duration)
                preview_offset = offset
                bpm_part_count = 0
                current_change_bpm_count += 1
            if i < start_measure or i == start_measure and j < start_beat:
                pass
            elif i > end_measure or i == end_measure and j > end_beat:
                if not is_end:
                    end_frame = offset_frame
                    is_end = True
            else:
                if not is_start:
                    start_frame = offset_frame
                    is_start = True
                rms[offset_frame:offset_frame+frame_length] = compute_rms(y_normalized[offset_frame:offset_frame+frame_length], frame_length, hop_length)
                beat_df.loc[part_count] = [i, j, offset, offset_frame, frame_quantize, "playing"]
            part_count += 1
            bpm_part_count += 1
    if not is_end:
        end_frame = full_frame
        is_end = True
    meta["preparation_part"] = preparation_finish
    beat_df.loc[:preparation_finish-1, "section"] = "preparation"
    meta["lingering_part"] = (part_count+1) - lingering_start
    beat_df.loc[lingering_start-1:, "section"] = "lingering"
    #print("---finish---")

    if max_rms is None:
        max_rms = np.max(rms)
    elif np.max(rms[start_frame:end_frame]) > max_rms:
        max_rms = np.max(rms[start_frame:end_frame])
    rms[start_frame:end_frame] /= max_rms
    #plot(y=y_normalized[start_frame:end_frame], wave=y[start_frame:end_frame], rms=rms[start_frame:end_frame], thre=True, beat_bar=beat_bar[start_frame:end_frame])

    return rms, beat_df

In [ ]:
def output_temp_wav(file, y):
    #display(IPython.display.Audio(y, rate=samplingrate))
    sf.write(file, y, samplingrate, format="WAV", subtype="FLOAT")
    print_outputlog(file)

In [ ]:
def whisper_transcribing(wavfile, make_force=False):
    duration = librosa.get_duration(path=wavfile)
    text = ""

    result = model.transcribe(wavfile)
    #print(result)
    if result["language"] != whisper_language:
        return text
    else:
        for segment in result["segments"]:
            if segment["start"] <= duration and segment["temperature"] <= threshold_whisper_temperature:
                text += segment["text"]
        if text != "":
            return text
        elif make_force:
            for segment in result["segments"]:
                if segment["start"] <= duration:
                    text += segment["text"]
            return text
        else:
            return text

In [ ]:
def transcribed_text_modify(text):
    text_modified = text.replace(" ", "、")
    if text_modified[-1] != "。":
        text_modified += "。"
    return text_modified

In [ ]:
def furigana_whisper_transcribing(audio_path, prompt):
    generate_kwargs = {"prompt_ids": pipe.tokenizer.get_prompt_ids(prompt, return_tensors="pt").to(pipe.device)}
    return pipe(audio_path, generate_kwargs=generate_kwargs)["text"]

In [ ]:
def _extract_katakana(text):
    return "".join(_katakana_re.findall(jaconv.hira2kata(text)))

def _extract_yomi(node):
    feat = node.feature
    kana = feat.kana
    pron = feat.pron
    pos1 = feat.pos1
    surf = node.surface

    if kana == "ハ" and pos1 == "助詞":
        return "ワ"
    if kana == "ヘ" and pos1 == "助詞":
        return "エ"
    if kana == "コンニチハ":
        return "コンニチワ"
    if kana == "コンバンハ":
        return "コンバンワ"
    if kana and kana != "*":
        return kana
    if pron and pron != "*":
        return pron
    if surf in {"。", "、"}:
        return surf
    return _extract_katakana(surf)

def _levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[m][n]

def get_best_match_yomi(text, yomi_pred, n=512):
    best_dist = float("inf")
    best_yomi = ""

    for nodes in tagger.nbestToNodeList(text, n):
        yomi_cand = "".join(_extract_yomi(nd) for nd in nodes)
        dist = _levenshtein(yomi_pred, yomi_cand)

        if dist < best_dist:
            best_dist = dist
            best_yomi = yomi_cand
            if dist == 0:
                break

    if best_dist == float("inf"):
        raise ValueError("No valid yomi candidates found.")
    best_dist = int(best_dist)

    return best_yomi

In [ ]:
def phoneme_converting(text):
    text = jaconv.kata2hira(text)
    phoneme = []
    text_length = len(text)
    text_number = 0
    letter = ""
    next_letter = ""
    while text_number < text_length:
        letter = text[text_number]
        if text_number < text_length-1:
            next_letter = text[text_number+1]
        else:
            next_letter = ""
        if next_letter in kana_sulky:
            if np.where(kana.values == letter)[1][0] == kana_sulky.index(next_letter):
                co, vo = np.where(kana.values == letter)
            else:
                co, vo = np.where(kana.values == letter+next_letter)
            text_number += 1
        else:
            co, vo = np.where(kana.values == letter)
        if letter == "、":
            if next_letter != "、":
                phoneme.append(phone_rest)
        elif letter == kana_con:
            phoneme.append(phone_con)
        elif letter == kana_N:
            phoneme.append(phone_N)
        else:
            if co[0] > 0:
                phoneme.append(kana.index[co[0]])
            phoneme.append(kana.columns[vo[0]])
        text_number += 1
    return " ".join(phoneme)

In [ ]:
def transcribe_lyric(wavfile, make_force=False):
    lyric = whisper_transcribing(wavfile, make_force=make_force)
    #print(lyric)
    if not lyric:
        return {}
    else:
        lyric_modified = transcribed_text_modify(lyric)
        #print(lyric_modified)
        lyric_furigana = furigana_whisper_transcribing(wavfile, lyric_modified)
        #print(lyric_furigana)
        if re.search(r"[0-9A-Za-z]", lyric_modified):
            lyric_furigana_modified = lyric_furigana
        else:
            lyric_furigana_modified = get_best_match_yomi(lyric_modified, lyric_furigana)
        #print(lyric_furigana_modified)
        lyric_phoneme = phoneme_converting(lyric_furigana_modified.replace("。", "").replace("ー", "").replace("ー", "").replace("・", ""))
        #print(lyric_phoneme)
        lyrics = {
            "lyric": lyric_modified,
            "furigana": lyric_furigana,
            "furigana_modified": lyric_furigana_modified,
            "phoneme": lyric_phoneme
        }
        return lyrics

In [ ]:
def output_temp_txt(file, texts):
    with open(file, mode="w") as f:
        f.write("\n".join(texts))
    print_outputlog(file)

In [ ]:
def transcribed_phoneme_modify(phoneme_str):
    phoneme_list = phoneme_str.split(" ")
    if phoneme_list[0] != phone_rest:
        phoneme_list.insert(0, phone_rest)
    if phoneme_list[-1] != phone_rest:
        phoneme_list.append(phone_rest)
    return " ".join(phoneme_list).replace("dw", "d").replace("fy", "f").replace("gw", "g").replace("jh", "j").replace("q", "k").replace("qy", "ky").replace("th", "t").replace("tw", "t").replace("vy", "v").replace("wh", "w").replace("zh", "z")

In [ ]:
def transcribe_label(wavfile, phoneme):
    x = librosa.load(wavfile, sr=16000, mono=True, dtype=np.float32)[0]
    label = alignmer.align(x, phoneme, min_aligned_timeframe)
    return label

In [ ]:
def output_temp_lab(file, label):
    with open(file, mode="w") as f:
        for lab in label:
            start, end, phoneme = lab
            start = math.floor(start*lab_time_magnification+0.5)
            end = math.floor(end*lab_time_magnification+0.5)
            phoneme = phoneme.replace("I", "i").replace("U", "u")
            f.write("{} {} {}\n".format(start, end, phoneme))
    print_outputlog(file)

In [ ]:
def output_temp(temp_dir, basename, y, make_force=False):
    global temp_id

    temp_wav_dir = os.path.join(temp_dir, "wav/")
    temp_wav_file = os.path.join(temp_wav_dir, "{}.wav").format(basename)
    temp_txt_dir = os.path.join(temp_dir, "txt/")
    temp_txt_file = os.path.join(temp_txt_dir, "{}.txt").format(basename)
    temp_lab_dir = os.path.join(temp_dir, "lab/")
    temp_lab_file = os.path.join(temp_lab_dir, "{}.lab").format(basename)

    output_temp_wav(temp_wav_file, y)
    lyrics = transcribe_lyric(temp_wav_file, make_force=make_force)
    if lyrics:
        is_successed = True
        output_temp_txt(temp_txt_file, list(lyrics.values()))
        phoneme_modified = transcribed_phoneme_modify(lyrics["phoneme"])
        label = transcribe_label(temp_wav_file, phoneme_modified)
        output_temp_lab(temp_lab_file, label)
        temp_id += 1
    else:
        is_successed = False
        delete_file(temp_wav_file)
    return is_successed

In [ ]:
def music_segmentation(meta, y, rms, beat_df, start_index=0, end_index=None):
    title = meta["title"]
    temp_dir = "./temp/{}/".format(title)

    if end_index is None:
        end_index = beat_df.shape[0]-1

    segment_start_index = start_index
    segment_start_time = 0
    segment_start_frame = 0
    segment_start_position = "1-1"
    segment_end_index = end_index
    segment_end_time = 0
    segment_end_frame = 0
    segment_end_position = "1-1"
    segment_min_mean_rms = 1.0
    is_segment = False
    is_in_thre = False
    current_section = ""
    beat_per_measure = 0

    for i in range(start_index, end_index):
        beat_row = beat_df.loc[i]
        if beat_row["beat_number"] == 1 or i == start_index:
            beat_per_measure = beat_df[beat_df["measure_number"] == beat_row["measure_number"]].shape[0]
        if beat_row["section"] == "preparation":
            current_section = "preparation"
        elif beat_row["section"] == "lingering":
            current_section = "lingering"
            mean_rms = np.mean(rms[beat_row["frame"]:beat_df.at[i+1, "frame"]])
            if mean_rms <= threshold_normalized_rms_minlimit:
                if is_segment:
                    segment_min_mean_rms = mean_rms
                    segment_end_index = i+1
                    segment_end_time = beat_df.at[i+1, "time"]
                    segment_end_frame = beat_df.at[i+1, "frame"]
                    segment_end_position = "{}-{}".format(beat_df.at[i+1, "measure_number"], beat_df.at[i+1, "beat_number"])
                if segment_end_time - segment_start_time <= max_temp_duration:
                    if segment_end_index - segment_start_index >= min_temp_measure*beat_per_measure:
                        basename = "{}_{:03}_{}".format(title, temp_id, int(segment_start_time*lab_time_magnification))
                        is_successed = output_temp(temp_dir, basename, y[segment_start_frame:segment_end_frame], make_force=True)
                        segment_min_mean_rms = 1.0
                        is_segment = False
                        is_in_thre = False
                        break
                    else:
                        is_segment = True
                else:
                    rms, beat_df = measurement_rms(meta, y, rms=rms, beat_df=beat_df, frame_quantize=frame_quantizes[frame_quantizes.index(beat_df.loc[segment_start_index:segment_end_index, "frame_quantize"].min())+1], start=segment_start_position, end=segment_end_position)
                    rms, beat_df = music_segmentation(meta, y, rms, beat_df, start_index=segment_start_index, end_index=segment_end_index)
                    segment_min_mean_rms = 1.0
                    is_segment = False
                    is_in_thre = False
                    break
            elif mean_rms <= segment_min_mean_rms:
                segment_min_mean_rms = mean_rms
                segment_end_index = i+1
                segment_end_time = beat_df.at[i+1, "time"]
                segment_end_frame = beat_df.at[i+1, "frame"]
                segment_end_position = "{}-{}".format(beat_df.at[i+1, "measure_number"], beat_df.at[i+1, "beat_number"])
                is_segment = False
                is_in_thre = True
        else:
            current_section = "playing"
            if np.sum(rms[beat_row["frame"]:beat_df.at[i+1, "frame"]] > threshold_normalized_rms) > (beat_df.at[i+1, "frame"]-beat_row["frame"])/segmentation_rate:  #有音区間
                if is_in_thre:  #区切りが未確定だった時
                    if segment_end_time - segment_start_time <= max_temp_duration:  #30秒ルールを満たしている時
                        if segment_end_index - segment_start_index >= min_temp_measure*beat_per_measure:  #1小節以上の長さがある時
                            basename = "{}_{:03}_{}".format(title, temp_id, int(segment_start_time*lab_time_magnification))
                            is_successed = output_temp(temp_dir, basename, y[segment_start_frame:segment_end_frame])
                            if is_successed:
                                is_segment = False
                        else:
                            is_segment = True
                    else:
                        rms, beat_df = measurement_rms(meta, y, rms=rms, beat_df=beat_df, frame_quantize=frame_quantizes[frame_quantizes.index(beat_df.loc[segment_start_index:segment_end_index, "frame_quantize"].min())+1], start=segment_start_position, end=segment_end_position)
                        rms, beat_df = music_segmentation(meta, y, rms, beat_df, start_index=segment_start_index, end_index=segment_end_index)
                        is_segment = False
                    segment_min_mean_rms = 1.0
                    is_in_thre = False
                if not is_segment:
                    if current_section == "preparation":
                        segment_start_index = i-1
                        segment_start_time = beat_df.at[i-1, "time"]
                        segment_start_frame = beat_df.at[i-1, "frame"]
                        segment_start_position = "{}-{}".format(beat_df.at[i-1, "measure_number"], beat_df.at[i-1, "beat_number"])
                    else:
                        segment_start_index = i
                        segment_start_time = beat_row["time"]
                        segment_start_frame = beat_row["frame"]
                        segment_start_position = "{}-{}".format(beat_row["measure_number"], beat_row["beat_number"])
                    is_segment = True
            else:  #無音区間
                if is_segment:  #区切りが未決定の時
                    mean_rms = np.mean(rms[beat_row["frame"]:beat_df.at[i+1, "frame"]])
                    if mean_rms <= threshold_normalized_rms_minlimit:
                        segment_min_mean_rms = mean_rms
                        segment_end_index = i+1
                        segment_end_time = beat_df.at[i+1, "time"]
                        segment_end_frame = beat_df.at[i+1, "frame"]
                        segment_end_position = "{}-{}".format(beat_df.at[i+1, "measure_number"], beat_df.at[i+1, "beat_number"])
                        is_segment = False  #区切りの確定
                    elif mean_rms <= segment_min_mean_rms:
                        segment_min_mean_rms = mean_rms
                        segment_end_index = i+1
                        segment_end_time = beat_df.at[i+1, "time"]
                        segment_end_frame = beat_df.at[i+1, "frame"]
                        segment_end_position = "{}-{}".format(beat_df.at[i+1, "measure_number"], beat_df.at[i+1, "beat_number"])
                    is_in_thre = True
    if is_segment or is_in_thre:
        segment_end_index = i+1
        segment_end_time = beat_df.at[i+1, "time"]
        segment_end_frame = beat_df.at[i+1, "frame"]
        segment_end_position = "{}-{}".format(beat_df.at[i+1, "measure_number"], beat_df.at[i+1, "beat_number"])
        if segment_end_time - segment_start_time <= max_temp_duration:
            basename = "{}_{:03}_{}".format(title, temp_id, int(segment_start_time*lab_time_magnification))
            is_successed = output_temp(temp_dir, basename, y[segment_start_frame:segment_end_frame], make_force=True)
        else:
            rms, beat_df = measurement_rms(meta, y, rms=rms, beat_df=beat_df, frame_quantize=frame_quantizes[frame_quantizes.index(beat_df.loc[segment_start_index:segment_end_index, "frame_quantize"].min())+1], start=segment_start_position, end=segment_end_position)
            rms, beat_df = music_segmentation(meta, y, rms, beat_df, start_index=segment_start_index, end_index=segment_end_index)
    return rms, beat_df

#### snapping_label

In [ ]:
def parse_filename(filename):
    result = re.compile(
        r'^(?P<title>[^_]+)'
        r'_'
        r'(?P<temp_id>[^_]+)'
        r'_'
        r'(?P<start_time>[^_]+)$'
    ).match(filename)
    if result is not None:
        title = result.group("title")
        temp_id = int(result.group("temp_id"))
        start_time = int(result.group("start_time"))
        return title, temp_id, start_time
    else:
        raise NotImplementedError("検索条件にマッチしませんでした。")

In [ ]:
def output_lab(file, label):
    with open(file, mode="w") as f:
        for lab in label:
            start, end, phoneme = lab
            f.write("{} {} {}\n".format(start, end, phoneme))
    print_outputlog(file)

In [ ]:
def merge_temp_lab(temp_lab_dir, duration):
    label = deque()
    label.append([0, 0, phone_rest])
    for temp_lab_file in glob.glob(os.path.join(temp_lab_dir, "*.lab")):
        start_time = parse_filename(os.path.splitext(os.path.basename(temp_lab_file))[0])[2]
        with open(temp_lab_file, mode="r") as f:
            for lab in f:
                start, end, phoneme = lab.split(" ")
                start, end = [t + start_time for t in map(int, [start, end])]
                phoneme = phoneme[:-1]
                if phoneme == phone_rest:
                    prev_lab = label.pop()
                    if prev_lab[2] == phone_rest:
                        label.append([prev_lab[0], end, phoneme])
                    else:
                        label.extend([prev_lab, [start, end, phoneme]])
                else:
                    label.append([start, end, phoneme])
    lab = label.pop()
    start, end, phoneme = lab
    end = math.floor(duration*lab_time_magnification)
    label.append([start, end, phoneme])
    return label

In [ ]:
def merge_temp_txt(temp_txt_dir):
    phoneme = deque()
    for temp_txt_file in glob.glob(os.path.join(temp_txt_dir, "*.txt")):
        with open(temp_txt_file, mode="r") as f:
            texts = f.readlines()
            phoneme.extend(list(texts[-1].replace(phone_rest, "").replace('\n', "").split()))
    return phoneme

In [ ]:
def snapping_label(meta, label_time, phoneme):
    title = meta["title"]
    measure = meta["measure"]
    quantize = meta["quantize"]

    beat = meta["beat"]
    beats_csv_path = "./meta/{}/beats.csv".format(title)
    if os.path.isfile(beats_csv_path):
        beats = pd.read_csv(beats_csv_path)
    else:
        beats = pd.DataFrame([[1, beat]], columns=["measure_number", "beat"])
    nume_beat, denomi_beat = map(int, re.findall(r'[0-9]+', beat))
    beat_part = quantize//denomi_beat
    measure_part = beat_part*nume_beat
    change_beat_count = beats.shape[0]

    bpm = meta["bpm"]
    bpms_csv_path = "./meta/{}/bpms.csv".format(title)
    if os.path.isfile(bpms_csv_path):
        bpms = pd.read_csv(bpms_csv_path)
    else:
        bpms = pd.DataFrame([[1, 1, bpm]], columns=["measure_number", "beat_number", "bpm"])
    duration = (60/bpm)*(4/quantize)
    change_bpm_count = bpms.shape[0]

    current_change_beat_count = 0
    current_change_bpm_count = 0
    preview_offset = 0
    offset = 0
    part_count = 0
    bpm_part_count = 0

    lab = []
    lab_start = 0
    lab_start_part = 0
    lab_end = 0
    lab_end_part = 0
    lab_phoneme = ""
    phoneme_count = 0
    label_beat = deque()
    for i in range(1, measure+1):
        #print("---measure number: {}---".format(i))
        if current_change_beat_count < change_beat_count and beats.iat[current_change_beat_count, 0] == i:
            beat = beats.iat[current_change_beat_count, 1]
            nume_beat, denomi_beat = map(int, re.findall(r'[0-9]+', beat))
            beat_part = quantize//denomi_beat
            measure_part = beat_part*nume_beat
            current_change_beat_count += 1
        for j in range(1, measure_part+1):
            offset = duration*bpm_part_count + preview_offset
            if current_change_bpm_count < change_bpm_count and bpms.iat[current_change_bpm_count, 0] == i and bpms.iat[current_change_bpm_count, 1] == j:
                bpm = bpms.iat[current_change_bpm_count, 2]
                duration = (60/bpm)*(4/quantize)
                preview_offset = offset
                bpm_part_count = 0
                current_change_bpm_count += 1
            part_count += 1
            bpm_part_count += 1
            #print("beat number: {} (t = {}s, part = {})".format(j, offset, part_count))
            while True:
                if label_time and not lab:
                    lab = label_time.popleft()
                    lab_start, lab_end, lab_phoneme = lab
                    lab_start /= lab_time_magnification
                    lab_end /= lab_time_magnification
                    if lab_phoneme != phone_rest:
                        lab_phoneme = phoneme[phoneme_count]
                        phoneme_count += 1
                    lab_start_part = part_count + float((((lab_start-offset)*10+duration/2)//duration)/10)
                    lab_end_part = lab_start_part + 1
                elif not label_time and not lab:
                    break
                if lab_end < offset+duration:
                    lab_end_part = part_count + float((((lab_end-offset)*10+duration/2)//duration)/10)
                    label_beat.append([lab_start_part, lab_end_part, lab_phoneme])
                    lab.clear()
                else:
                    break
    if lab:
        label_beat.append([lab_start_part, part_count+1, lab_phoneme])
    return label_beat

#### music_transcription

In [ ]:
def compute_pitch(y, frame_length, hop_length):
    f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=samplingrate, frame_length=frame_length, hop_length=hop_length)
    return f0

In [ ]:
def mode_1darray(array, reference_value):
    values, counts = np.unique(array, return_counts=True)
    mode_value_indexes = [i for i, x in enumerate(counts) if x == np.max(counts)]
    if len(mode_value_indexes) == 1:
        mode_value = values[mode_value_indexes[0]]
    else:
        mode_values = [values[index] for index in mode_value_indexes]
        mode_values_diff = [abs(value-reference_value) for value in mode_values]
        mode_value_indexes = [i for i, x in enumerate(mode_values_diff) if x == np.min(mode_values_diff)]
        if len(mode_value_indexes) == 1:
            mode_value = mode_values[mode_value_indexes[0]]
        else:
            mode_values = [mode_values[index] for index in mode_value_indexes]
            mode_value = np.median(mode_values)
    return mode_value

In [ ]:
def note_transcription(y, frame_length, hop_length, note_length, quantize, prev_pitch):
    note_numbers = np.array([])
    rest_start = None
    num_separation = max_quantize // quantize
    f0 = compute_pitch(y, frame_length, hop_length)
    for i in range(note_length):
        sub_f0 = f0[num_separation*i:num_separation*(i+1)]
        if np.sum(np.isnan(sub_f0)) >= len(sub_f0)/2:
            if rest_start is None:
                rest_start = i
        else:
            if rest_start is not None:
                rest_start = None
        note_numbers = np.append(note_numbers, [math.floor(len(scale_names)*math.log2(pitch/A_lowest)+0.5) for pitch in sub_f0[~np.isnan(sub_f0)]])
    note_number = rest_number if rest_start==0 else int(mode_1darray(note_numbers, prev_pitch))
    return note_number, rest_start

In [ ]:
def music_transcription(meta, y, label_beat):
    title = meta["title"]
    measure = meta["measure"]
    quantize = meta["quantize"]

    beat = meta["beat"]
    beats_csv_path = "./meta/{}/beats.csv".format(title)
    if os.path.isfile(beats_csv_path):
        beats = pd.read_csv(beats_csv_path)
    else:
        beats = pd.DataFrame([[1, beat]], columns=["measure_number", "beat"])
    nume_beat, denomi_beat = map(int, re.findall(r'[0-9]+', beat))
    beat_part = quantize//denomi_beat
    measure_part = beat_part*nume_beat
    change_beat_count = beats.shape[0]

    bpm = meta["bpm"]
    bpms_csv_path = "./meta/{}/bpms.csv".format(title)
    if os.path.isfile(bpms_csv_path):
        bpms = pd.read_csv(bpms_csv_path)
    else:
        bpms = pd.DataFrame([[1, 1, bpm]], columns=["measure_number", "beat_number", "bpm"])
    change_bpm_count = bpms.shape[0]

    duration = (60/bpm)*(4/quantize)
    frame_duration = (60/bpm)*(4/quantize)
    frame_length = int(samplingrate*frame_duration)
    hop_duration = (60/bpm)*(4/max_quantize)
    hop_length = int(samplingrate*hop_duration)

    full_duration = librosa.get_duration(y=y, sr=samplingrate)
    full_frame = y.shape[0]

    current_change_beat_count = 0
    current_change_bpm_count = 0
    preview_offset = 0
    offset = 0
    offset_frame = 0
    part_count = 0
    bpm_part_count = 0

    label = []
    label_start = 1
    label_start_frame = 0
    label_end = 1
    label_end_frame = 0
    label_length = 1
    label_phoneme = ""
    note = []
    note_number = rest_number
    prev_note_number = 39
    note_phoneme = ""
    temp_note_phoneme = ""
    note_start = 1
    note_end = 1

    notes = deque([[rest_number, phone_rest, 1, 1]])
    for i in range(1, measure+1):
        #print("---measure number: {}---".format(i))
        if current_change_beat_count < change_beat_count and beats.iat[current_change_beat_count, 0] == i:
            beat = beats.iat[current_change_beat_count, 1]
            nume_beat, denomi_beat = map(int, re.findall(r'[0-9]+', beat))
            beat_part = quantize//denomi_beat
            measure_part = beat_part*nume_beat
            current_change_beat_count += 1
        for j in range(1, measure_part+1):
            offset = duration*bpm_part_count + preview_offset
            offset_frame = int((offset/full_duration)*full_frame)
            if current_change_bpm_count < change_bpm_count and bpms.iat[current_change_bpm_count, 0] == i and bpms.iat[current_change_bpm_count, 1] == j:
                bpm = bpms.iat[current_change_bpm_count, 2]
                duration = (60/bpm)*(4/quantize)
                frame_duration = (60/bpm)*(4/quantize)
                frame_length = int(samplingrate*frame_duration)
                hop_duration = (60/bpm)*(4/max_quantize)
                hop_length = int(samplingrate*hop_duration)
                preview_offset = offset
                bpm_part_count = 0
                current_change_bpm_count += 1
            part_count += 1
            bpm_part_count += 1
            #print("beat number: {} (t = {}s, frame = {}, part={})".format(j, offset, offset_frame, part_count))
            while True:
                if label_beat and not label:
                    label = label_beat.popleft()
                    label_start, label_end, label_phoneme = label
                    label_start_frame = offset_frame+frame_length*math.floor(label_start-part_count+0.5)
                    label_length = max(1, math.floor(label_end+0.5)-math.floor(label_start+0.5))
                elif not label_beat and not label:
                    break
                if part_count+1 >= label_end:
                    if label_phoneme in phone_vowel:  #音素が母音の時
                        label_start_frame = min(label_start_frame, offset_frame)
                        label_end_frame = offset_frame+frame_length if label_length <= 1 else offset_frame+int(frame_length*(label_end-part_count))
                        note_number, rest_start = note_transcription(y[label_start_frame:label_end_frame], frame_length, hop_length, label_length, quantize, prev_note_number)
                        #print(note_phoneme+label_phoneme, note_number)
                        prev_note = notes.pop()
                        if note_number != prev_note[0] or len(note_phoneme) != 0:
                            if math.floor(prev_note[3]) == math.floor(label_start+0.5):
                                prev_note = [prev_note[0], prev_note[1], prev_note[2], float(int(label_start))]
                                note_start = float(int(label_start))
                            else:
                                note_start = label_start
                            notes.append(prev_note)
                            note_phoneme += label_phoneme
                            if temp_note_phoneme:
                                note_phoneme = temp_note_phoneme+"-"+note_phoneme
                                temp_note_phoneme = ""
                        else:
                            if temp_note_phoneme:
                                note_phoneme = prev_note[1]+"-"+temp_note_phoneme+"-"+label_phoneme if prev_note[1] != phone_rest else temp_note_phoneme+"-"+label_phoneme
                                temp_note_phoneme = ""
                            else:
                                note_phoneme = prev_note[1]+"-"+label_phoneme if prev_note[1] != phone_rest else label_phoneme
                            note_start = prev_note[2]
                        if rest_start is not None:
                            if rest_start == 0:
                                rest_start = note_start
                                rest_end = label_end
                                note_rest = [rest_number, phone_rest, rest_start, rest_end]
                                notes.append(note_rest)
                                temp_note_phoneme = note_phoneme
                            else:
                                note_end = float(int(label_start+rest_start))
                                rest_start = float(int(label_start+rest_start))
                                rest_end = label_end
                                note = [note_number, note_phoneme, note_start, note_end]
                                note_rest = [rest_number, phone_rest, rest_start, rest_end]
                                notes.extend([note, note_rest])
                        else:
                            note_end = label_end
                            note = [note_number, note_phoneme, note_start, note_end]
                            notes.append(note)
                        note_phoneme = ""
                        if note_number < rest_number:
                            prev_note_number = note_number
                    elif label_phoneme == phone_con:  #音素が促音の時
                        prev_note = notes.pop()
                        note_phoneme = prev_note[1]+"-"+label_phoneme if prev_note[1] != phone_rest else prev_note[1]
                        note_end = label_end
                        note = [prev_note[0], note_phoneme, prev_note[2], note_end]
                        notes.append(note)
                        note_phoneme = ""
                    elif label_phoneme == phone_N:  #音素が撥音の時
                        label_start_frame = min(label_start_frame, offset_frame)
                        label_end_frame = offset_frame+frame_length if label_length <= 1 else offset_frame+int(frame_length*(label_end-part_count))
                        note_number, rest_start = note_transcription(y[label_start_frame:label_end_frame], frame_length, hop_length, label_length, quantize, prev_note_number)
                        #print(note_phoneme+label_phoneme, note_number)
                        prev_note = notes.pop()
                        if note_number != prev_note[0] or len(note_phoneme) != 0:
                            if math.floor(prev_note[3]) == math.floor(label_start+0.5):
                                prev_note = [prev_note[0], prev_note[1], prev_note[2], float(int(label_start))]
                                note_start = float(int(label_start))
                            else:
                                note_start = label_start
                            notes.append(prev_note)
                            note_phoneme += label_phoneme
                            if temp_note_phoneme:
                                note_phoneme = temp_note_phoneme+"-"+note_phoneme
                                temp_note_phoneme = ""
                        else:
                            if temp_note_phoneme:
                                note_phoneme = prev_note[1]+"-"+temp_note_phoneme+"-"+label_phoneme if prev_note[1] != phone_rest else temp_note_phoneme+"-"+label_phoneme
                                temp_note_phoneme = ""
                            else:
                                note_phoneme = prev_note[1]+"-"+label_phoneme if prev_note[1] != phone_rest else label_phoneme
                            note_start = prev_note[2]
                        if rest_start is not None:
                            if rest_start == 0:
                                rest_start = note_start
                                rest_end = label_end
                                note_rest = [rest_number, phone_rest, rest_start, rest_end]
                                notes.append(note_rest)
                                temp_note_phoneme = note_phoneme
                            else:
                                note_end = float(int(label_start+rest_start))
                                rest_start = float(int(label_start+rest_start))
                                rest_end = label_end
                                note = [note_number, note_phoneme, note_start, note_end]
                                note_rest = [rest_number, phone_rest, rest_start, rest_end]
                                notes.extend([note, note_rest])
                        else:
                            note_end = label_end
                            note = [note_number, note_phoneme, note_start, note_end]
                            notes.append(note)
                        note_phoneme = ""
                        if note_number < rest_number:
                            prev_note_number = note_number
                    elif label_phoneme == phone_rest:  #音素がポーズの時
                        note_number = rest_number
                        note_phoneme = phone_rest
                        prev_note = notes.pop()
                        if prev_note[0] != rest_number:
                            notes.append(prev_note)
                            note_start = label_start
                        note_end = label_end
                        note = [note_number, note_phoneme, note_start, note_end]
                        notes.append(note)
                        note_phoneme = ""
                    else:  #音素が子音の時
                        prev_note = notes.pop()
                        note_end = label_end
                        note = [prev_note[0], prev_note[1], prev_note[2], note_end]
                        notes.append(note)
                        note_phoneme = label_phoneme
                    label.clear()
                else:
                    break
    return notes

#### making_score

In [ ]:
def kana_conversion(roma_text):
    if roma_text in {phone_con, phone_N}:
        kana_text = kana_con if roma_text == phone_con else kana_N
    else:
        kana_text = kana.at[float('nan'), roma_text.lower()] if len(roma_text) == 1 else kana.at[roma_text[:-1], roma_text[-1]]
    return kana_text

In [ ]:
def making_score(notes):
    temp_pitch = ""
    temp_lyric = ""
    score = deque()
    for note in notes:
        pitch, lyric, note_start, note_end = note
        pitch = rest if pitch>=rest_number else "{}{}".format(scale_names[pitch%num_scale], (pitch+9)//num_scale)
        lyric = "" if lyric == phone_rest else "".join(list(map(kana_conversion, lyric.split("-"))))
        note_length = math.floor(note_end+0.5) - math.floor(note_start+0.5)
        #print([pitch, lyric, note_length], [temp_pitch, temp_lyric])
        if note_length <= 0:
            if (note_end*10)%10 >= 5:
                prev_note = score.pop()
                prev_pitch, prev_lyric, prev_note_length = prev_note
                if temp_lyric:
                    lyric = temp_lyric + lyric
                if pitch == prev_pitch or pitch == rest:
                    prev_lyric += lyric
                    score.append([prev_pitch, prev_lyric, prev_note_length])
                else:
                    prev_note_length -= 1
                    note_length = 1
                    score.extend([[prev_pitch, prev_lyric, prev_note_length], [pitch, lyric, note_length]])
            else:
                if pitch != rest:
                    temp_pitch = pitch
                temp_lyric += lyric
        else:
            if temp_pitch:
                if pitch == rest:
                    score.append([temp_pitch, temp_lyric, 1])
                    note_length -= 1
                else:
                    lyric = temp_lyric + lyric
                temp_pitch = ""
                temp_lyric = ""
            score.append([pitch, lyric, note_length])
    return score

#### score_to_musicxml

In [ ]:
def make_musicxml_head(title=None):
    musicxml_head = '''<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE score-partwise PUBLIC "-//Recordare//DTD MusicXML 4.0 Partwise//EN" "http://www.musicxml.org/dtds/partwise.dtd">
<score-partwise version="4.0">
'''
    if title is not None:
        musicxml_head += f'''<work>
<work-title>{title}</work-title>
</work>'''
    musicxml_head += '''
<part-list>
<score-part id="P1">
<part-name>MusicXML</part-name>
</score-part>
</part-list>
<part id="P1">
'''
    return musicxml_head

In [ ]:
def make_musicxml_attributes(divisions=None, key=None, beats=None, beat_type=None, clef=None):
    musicxml_attributes = '''<attributes>
'''
    if divisions is not None:  #四分音符の構成数の設定（原則1小節目のみで設定）
        musicxml_attributes += f'''<divisions>{divisions}</divisions>
'''
    if key is not None:  #調の設定
        musicxml_attributes += f'''<key>
<fifths>{key_numbers[key]}</fifths>
<mode>{"minor" if key[-1]=="m" else "major"}</mode>
</key>
'''
    if beats is not None and beat_type is not None:  #拍子の設定
        musicxml_attributes += f'''<time>
<beats>{beats}</beats>
<beat-type>{beat_type}</beat-type>
</time>
'''
    if clef is not None:  #音部記号の設定
        musicxml_attributes += f'''<clef>
<sign>{clef}</sign>
'''
        if clef == "G":
            musicxml_attributes += '''<line>2</line>
'''
        elif clef == "F":
            musicxml_attributes += '''<line>4</line>
'''
        musicxml_attributes += f'''</clef>
'''
    musicxml_attributes += '''</attributes>
'''
    return musicxml_attributes

In [ ]:
def make_musicxml_direction(bpm):  #その他の設定項目: 強弱記号（クレッシェンド/ディミヌエンド）
    musicxml_direction = f'''<direction>
<direction-type>
<metronome>
<beat-unit>quarter</beat-unit>
<per-minute>{bpm}</per-minute>
</metronome>
</direction-type>
<sound tempo="{bpm}" />
</direction>
'''
    return musicxml_direction

In [ ]:
def make_musicxml_notations(dynamics=None, tie_stop=False, tie_start=False):  #その他の設定項目: スラー、アーティキュレーション（スタッカート/テヌート）
    musicxml_notations = '''<notations>
'''
    if dynamics is not None:
        musicxml_notations += f'''<dynamics>
<{dynamics} />
</dynamics>
'''
    if tie_stop is True:
        musicxml_notations += '''<tied type="stop" />
'''
    if tie_start is True:
        musicxml_notations += '''<tied type="start" />
'''
    musicxml_notations += '''</notations>
'''
    return musicxml_notations

In [ ]:
def make_musicxml_note(note: list, dynamics=None, tie=""):
    pitch, lyric, duration = note
    
    musicxml_note = '''<note>
'''
    if pitch == rest:
        musicxml_note += '''<rest />
'''
    else:
        step, alter, octave = pitch[:-1], 0, pitch[-1]
        if len(step) != 1:  #音階に臨時記号が付いている時
            if step[-1] == "#":
                alter = 1
            elif step[-1] == "b":
                alter = -1
            step = step[:-1]
        musicxml_note += f'''<pitch>
<step>{step}</step>
<alter>{alter}</alter>
<octave>{octave}</octave>
</pitch>
'''
    musicxml_note += f'''<duration>{duration}</duration>
'''
    if dynamics is not None:
        if tie == "begin":
            musicxml_notations = make_musicxml_notations(dynamics=dynamics, tie_start=True)
        elif tie == "middle":
            musicxml_notations = make_musicxml_notations(dynamics=dynamics, tie_start=True, tie_stop=True)
        elif tie == "end":
            musicxml_notations = make_musicxml_notations(dynamics=dynamics, tie_stop=True)
        else:
            musicxml_notations = make_musicxml_notations(dynamics=dynamics)
        musicxml_note += musicxml_notations
    elif tie:
        if tie == "begin":
            musicxml_notations = make_musicxml_notations(tie_start=True)
        elif tie == "middle":
            musicxml_notations = make_musicxml_notations(tie_start=True, tie_stop=True)
        elif tie == "end":
            musicxml_notations = make_musicxml_notations(tie_stop=True)
        musicxml_note += musicxml_notations
    if pitch != rest:
        musicxml_note += '''<lyric>
'''
        if tie:
            musicxml_note += f'''<syllabic>{tie}</syllabic>
'''
        else:
            musicxml_note += '''<syllabic>single</syllabic>
'''
        musicxml_note += f'''<text>{lyric}</text>
</lyric>
'''
    musicxml_note += '''</note>
'''
    return musicxml_note

In [ ]:
def make_musicxml_body(meta, score, bpms, beats):
    bpm = 120  #bpmの初期化 (4分音符での表記)
    quantize = meta["quantize"]  #楽曲クオンタイズの取得
    divisions = meta["divisions"]  #四分音符の構成数の設定
    beat = "4/4"  #拍子の初期化
    nume_beats, denomi_beats = map(int, re.findall(r'[0-9]+', beat))  #拍子を分母(denomi)と分子(nume)に分解
    num_measure = meta["measure"]  #総小節数

    attributes = False  #attributesを挿入するフラグ
    direction = False  #directionを挿入するフラグ
    tie = ""  #タイのステータスを入れるところ
    change_beat_count = beats.shape[0]  #拍子が変化する回数を入れるところ
    current_change_beat_count = 0  #拍子が変化した回数を入れるところ

    musicxml_body = ''
    for n in range(1, num_measure+1):
        current_beat_number = 1  #現在の拍のポジションを入れるところ
        musicxml_body += f'''<measure number="{n}">
'''
        if current_change_beat_count < change_beat_count and beats.iat[current_change_beat_count, 0] == n:  #拍子変更を行う時
            beat = beats.iat[current_change_beat_count, 1]  #拍子の再設定
            nume_beats, denomi_beats = map(int, re.findall(r'[0-9]+', beat))
            current_change_beat_count += 1
            attributes = True
        if attributes:  #attributesの挿入フラグがオンになっている時
            if n == 1:  #1小節目の時
                musicxml_attributes = make_musicxml_attributes(divisions=divisions, key="C", beats=nume_beats, beat_type=denomi_beats, clef="G")
            else:
                musicxml_attributes = make_musicxml_attributes(beats=nume_beats, beat_type=denomi_beats)
            musicxml_body += musicxml_attributes
            attributes = False
        measure_duration = int(divisions*nume_beats*(4/denomi_beats))  #小節内に入るduration数
        measure_bpms = bpms[bpms["measure_number"] == n]  #小節内でのbpm変化のDataFrame
        measure_change_bpm_count = measure_bpms.shape[0]  #小節内でbpmが変化する回数を入れるところ
        measure_current_change_bpm_count = 0  #小節内でbpmが変化した回数を入れるところ
        while current_beat_number <= measure_duration:
            if measure_current_change_bpm_count < measure_change_bpm_count and measure_bpms.iat[measure_current_change_bpm_count, 1] == current_beat_number:  #bpm変更を行う時
                bpm = measure_bpms.iat[measure_current_change_bpm_count, 2]  #bpmの再設定
                measure_current_change_bpm_count += 1
                direction = True
            if direction:  #directionの挿入フラグがオンになっている時
                musicxml_body += make_musicxml_direction(bpm)
                direction = False
            left_duration = measure_duration - current_beat_number + 1  #小節内に入る残りのduration数
            note = score.popleft()
            if note[2] > left_duration:  #ノートのdurationが小節内の残りdurationより大きい時
                if note[0] != rest:  #ノートが休符でない時
                    tie = "begin" if not tie else "middle"
                next_note = [note[0], long_vowel, note[2]-left_duration]
                score.appendleft(next_note)
                note[2] = left_duration
            elif measure_current_change_bpm_count < measure_change_bpm_count:  #bpm変化が起こる可能性がある時
                beat_number = measure_bpms.iat[measure_current_change_bpm_count, 1]  #bpmが変化するポジションを入れるところ
                left_duration = beat_number - current_beat_number  #bpmが変化するまでの残りのduration数
                if note[2] > left_duration:  #ノートの途中でbpm変化が起こる時
                    next_note = [note[0], long_vowel, note[2]-left_duration]
                    score.appendleft(next_note)
                    note[2] = left_duration
            elif tie:  #タイにパラメータが設定されている時
                tie = "end"
            musicxml_note = make_musicxml_note(note, tie=tie)
            if tie == "end":  #タイのパラメータがendである時
                tie = ""
            musicxml_body += musicxml_note
            current_beat_number += note[2]
        musicxml_body += '''</measure>
'''
    return musicxml_body

In [ ]:
def make_musicxml_tail():
    musicxml_tail = '''</part>
</score-partwise>'''
    return musicxml_tail

In [ ]:
def score_to_musicxml(meta, score):
    title = meta["title"]
    beat = meta["beat"]
    beats_csv_path = "./meta/{}/beats.csv".format(title)
    if os.path.isfile(beats_csv_path):
        beats = pd.read_csv(beats_csv_path)
    else:
        beats = pd.DataFrame([[1, beat]], columns=["measure_number", "beat"])
    bpm = meta["bpm"]
    bpms_csv_path = "./meta/{}/bpms.csv".format(title)
    if os.path.isfile(bpms_csv_path):
        bpms = pd.read_csv(bpms_csv_path)
    else:
        bpms = pd.DataFrame([[1, 1, bpm]], columns=["measure_number", "beat_number", "bpm"])

    musicxml = ""
    musicxml += make_musicxml_head(title=title)
    musicxml += make_musicxml_body(meta, score, bpms, beats)
    musicxml += make_musicxml_tail()
    return musicxml

In [ ]:
def output_musicxml(file, musicxml):
    with open(file, mode="w", encoding="utf-8") as f:
        f.write(musicxml)
    print_outputlog(file)

#### main

In [ ]:
def wav_to_musicxml(meta):
    global min_y, max_y, max_rms, temp_id
    max_rms = None
    temp_id = 0

    title = meta["title"]

    temp_dir = "./temp/{}/".format(title)  #tempディレクトリのフォルダパスの設定
    if os.path.isdir(temp_dir):
        shutil.rmtree(temp_dir)
    for name in ["wav", "lab", "txt"]:
        os.makedirs(os.path.join(temp_dir, name))

    audio_path = "./audio/{}.wav".format(title)
    y = librosa.load(audio_path, sr=samplingrate)[0]
    duration = librosa.get_duration(y=y, sr=samplingrate)

    label_path = "./lab/predict/{}.lab".format(title)
    musicxml_path = "./musicxml/predict/{}.musicxml".format(title)

    min_y = np.min(np.abs(y))
    max_y = np.max(np.abs(y))

    rms, beat_df = measurement_rms(meta, y)
    rms, beat_df = music_segmentation(meta, y, rms, beat_df)
    label_time = merge_temp_lab(os.path.join(temp_dir, "lab/"), duration)
    #print(label_time)
    output_lab(label_path, label_time)
    phoneme = merge_temp_txt(os.path.join(temp_dir, "txt/"))
    #print(phoneme)
    label_beat = snapping_label(meta, label_time, phoneme)
    #print(label_beat)
    notes = music_transcription(meta, y, label_beat)
    #print(notes)
    score = making_score(notes)
    #print(score)
    musicxml = score_to_musicxml(meta, score.copy())
    #print(musicxml)
    output_musicxml(musicxml_path, musicxml)
    return score, musicxml

#### musicxml to score

In [ ]:
def musicxml_to_score(musicxml_path):
    divisions = 0
    durations = 0
    score = deque([[rest, "", 0, False]])

    with open(musicxml_path, "r", encoding="utf-8") as f:
        pitch, lyric, duration = "", "", 0
        is_append = False
        for line in f:
            line = re.sub("\s+", "", line)
            if re.match(r"<divisions>", line):
                divisions = int(re.search(r"(<divisions>)([0-9]+)(</divisions>)", line).groups()[1])
            elif re.match(r"<rest", line):
                pitch = rest
                lyric = ""
            elif re.match(r"<step>", line):
                pitch = re.search(r"(<step>)([A-G])(</step>)", line).groups()[1]
            elif re.match(r"<alter>", line):
                alter = int(re.search(r"(<alter>)(-?[0-1])(</alter>)", line).groups()[1])
                if alter == 1:
                    pitch += "#"
                elif alter == -1:
                    pitch += "b"
            elif re.match(r"<octave>", line):
                pitch += re.search(r"(<octave>)([0-8])(</octave>)", line).groups()[1]
            elif re.match(r"<duration>", line):
                duration = int(re.search(r"(<duration>)([0-9]+)(</duration>)", line).groups()[1])
                durations += duration
            elif re.match(r"<tie", line):
                if re.match(r'<tiedtype="(middle|stop)"/>', line):
                    pitch = "tie"
            elif re.match(r"<text>", line):
                lyric = re.search(r"(<text>)(.+)(</text)", line).groups()[1].replace("’", "").replace(".", "")

            if pitch == rest and duration != 0:
                prev_note = score.pop()
                if prev_note[0] == rest:
                    duration += prev_note[2]
                else:
                    score.append(prev_note)
                note = [pitch, lyric, duration, False]
                is_append = True
            elif pitch == "tie" and duration != 0:
                note = score.pop()
                note[2] += duration
                is_append = True
            elif lyric != "" and pitch == "":
                lyric = ""
            elif pitch != "" and lyric != "" and duration != 0:
                note = [pitch, lyric, duration, False]
                is_append = True

            if is_append:
                score.append(note)
                pitch, lyric, duration = "", "", 0
                is_append = False

    score = {
        "divisions": divisions,
        "durations": durations,
        "score": score
    }
    return score

#### svp to musicxml

In [ ]:
def svp_to_score(svp_path):
    with open(svp_path, encoding="utf-8", mode="r") as f:
        svp = json.load(f)
    notes = svp["library"][0]["notes"]
    
    prev_onset = 0
    prev_duration = 0
    score = deque()
    for i in range(len(notes)):
        note = notes[i]
        onset = note["onset"]
        duration = note["duration"]
        if onset > prev_onset+prev_duration:
            score.append([rest, "", (onset-(prev_onset+prev_duration))])
        pitch = note["pitch"]
        pitch -= SV2_pitch_correction
        lyric = note["lyrics"]
        score.append(["{}{}".format(scale_names[pitch%num_scale], (pitch+9)//num_scale), lyric, duration])
        prev_onset = onset
        prev_duration = duration
    score.pop()
    return score

In [ ]:
def svp_to_musicxml(svp_path, musicxml_path, meta, divisions=705600000):
    meta["divisions"] = divisions
    score = svp_to_score(svp_path)
    #print(score)
    musicxml = score_to_musicxml(meta, score.copy())
    #print(musicxml)
    output_musicxml(musicxml_path, musicxml)
    return score, musicxml

#### evaluation

In [ ]:
def evaluate_score(actual_musicxml_path, pred_musicxml_path):
    actual_score = musicxml_to_score(actual_musicxml_path)
    #print("---actual---")
    #print(actual_score)
    #print("------------")
    pred_score = musicxml_to_score(pred_musicxml_path)
    #print("----pred----")
    #print(pred_score)
    #print("------------")

    actual_score_divisions = actual_score["divisions"]
    pred_score_divisions = pred_score["divisions"]
    lcm_divisions = math.lcm(actual_score_divisions, pred_score_divisions)
    actual_score_rate = lcm_divisions / actual_score_divisions
    pred_score_rate = lcm_divisions / pred_score_divisions
    #print(actual_score["durations"]*actual_score_rate, pred_score["durations"]*pred_score_rate)
    actual_score = actual_score["score"]
    pred_score = pred_score["score"]

    num_note = 0
    correct_note = 0
    num_duration = 0
    correct_pitch_duration = 0
    actual_lyric = ""
    pred_lyric = ""
    actual_note = []
    pred_note = []
    while actual_score or pred_score:
        if not actual_note:
            actual_note = actual_score.popleft()
            actual_note[2] *= actual_score_rate
        if not pred_note:
            pred_note = pred_score.popleft()
            pred_note[2] *= pred_score_rate

        length = min(actual_note[2], pred_note[2])
        if actual_note[3] == False:
            if pred_note[0] == actual_note[0] and pred_note[1] == actual_note[1] and actual_note[2] == pred_note[2] and pred_note[3] == False:
                correct_note += 1
            num_note += 1
        if pred_note[0] == actual_note[0]:
            correct_pitch_duration += length
        num_duration += length
        if actual_note[3] == False:
            actual_lyric += actual_note[1]
            actual_note[3] = True
        if pred_note[3] == False:
            pred_lyric += pred_note[1]
            pred_note[3] = True

        actual_note[2] -= length
        if actual_note[2] <= 0:
            actual_note.clear()
        pred_note[2] -= length
        if pred_note[2] <= 0:
            pred_note.clear()

    note_accuracy = correct_note/num_note
    pitch_accuracy = correct_pitch_duration/num_duration
    lyric_cer = process_characters(actual_lyric, pred_lyric).cer
    return note_accuracy, pitch_accuracy, lyric_cer

#### execution

In [ ]:
def record_result(csv_path, index, result):
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
    else:
        df = pd.DataFrame(columns=["title", "pred_note_accuracy", "pred_pitch_accuracy", "pred_lyric_cer", "SV2_note_accuracy", "SV2_pitch_accuracy", "SV2_lyric_cer", "recorded_date"], dtype={"title": object})
    df.loc[index] = result
    df.to_csv(csv_path, index=False)
    print_outputlog(csv_path)

In [ ]:
def execution(meta_file, csv_path, start_index=0, end_index=None):
    metas = pd.read_csv(meta_file, dtype={"title": object})
    #display(metas)

    if end_index is None:
        end_index = metas.shape[0]

    for i in range(start_index, end_index):
        row = metas.iloc[i, :]
    
        title = row["title"]
    
        meta = {
            "title": title,
            "bpm": row["bpm"],
            "measure": row["measure"],
            "beat": row["beat"],
            "quantize": row["quantize"],
            "divisions": max(row["quantize"]//4, 1),
            "preparation": row["preparation"],
            "lingering": row["lingering"],
        }
    
        score, musicxml = wav_to_musicxml(meta)
        #print(score)
        #print(musicxml)

        musicxml_dir = "./musicxml/"
        musicxml_file = "{}.musicxml".format(title)
        original_musicxml_path = os.path.join(musicxml_dir, "original/", musicxml_file)
        pred_musicxml_path = os.path.join(musicxml_dir, "predict/", musicxml_file)
        pred_note_accuracy, pred_pitch_accuracy, pred_lyric_cer = evaluate_score(original_musicxml_path, pred_musicxml_path)
        svp_path = "./svp/{}.svp".format(title)
        SV2_musicxml_path = os.path.join(musicxml_dir, "SV2/", musicxml_file)
        svp_to_musicxml(svp_path, SV2_musicxml_path, meta, divisions=SV2_divisions)
        SV2_note_accuracy, SV2_pitch_accuracy, SV2_lyric_cer = evaluate_score(original_musicxml_path, SV2_musicxml_path)
        record_result(csv_path, i, [title, pred_note_accuracy, pred_pitch_accuracy, pred_lyric_cer, SV2_note_accuracy, SV2_pitch_accuracy, SV2_lyric_cer, datetime.now().strftime("%Y/%m/%d %H:%M:%S")])

#### Analysis

In [ ]:
def plot_scatter(ser1, ser2, xlabel="", ylabel="", title=""):
    plt.figure(figsize=(4, 4))
    plt.scatter(ser1, ser2)
    plt.plot(np.arange(0.0, 1.0, 10**(-4)), np.arange(0.0, 1.0, 10**(-4)), color="orange")
    plt.xlabel(xlabel)
    plt.xlim(0, 1)
    plt.ylabel(ylabel)
    plt.ylim(0, 1)
    plt.title(title)
    plt.savefig("{}.png".format(title), dpi=360)
    plt.show()
    print_outputlog("{}.png".format(title))

In [ ]:
def ttest(ser1, ser2, xlabel=None, ylabel=None, a=0.05, reverse=False):
    t, p = ss.ttest_ind(ser1, ser2, equal_var=False)
    if reverse:
        t *= -1
    if t<0 and p/2<a:
        print("{} is statistically more significant than {}. (p={:.5})".format(ylabel if ylabel is not None else ser2.name, xlabel if xlabel is not None else ser1.name, p))
    elif t>0 and p/2<a:
        print("{} is statistically more significant than {}. (p={:.5})".format(xlabel if xlabel is not None else ser1.name, ylabel if ylabel is not None else ser2.name, p))
    else:
        print("{} and {} didn't differ significantly. (p={:.5})".format(xlabel if xlabel is not None else ser1.name, ylabel if ylabel is not None else ser2.name, p))

In [ ]:
def analysis_ser(ser1, ser2, xlabel, ylabel, title, reverse=False):
    plot_scatter(ser1, ser2, xlabel=xlabel, ylabel=ylabel, title=title)
    ttest(ser1, ser2, xlabel=xlabel, ylabel=ylabel, reverse=reverse)

In [ ]:
def plot_cluster_scatter(ser1, cls, n_clusters, ser2=None, start_number=1, xlabel="", ylabel="", title=""):
    for i in range(start_number, n_clusters+start_number):
        ser1_cls = ser1[cls==i]
        if ser2 is None:
            ser2_cls = np.zeros(ser1_cls.shape[0])
        else:
            ser2_cls = ser2[cls==i]
        plt.title(title)
        plt.scatter(ser1_cls, ser2_cls, label="cluster{}".format(i))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()

In [ ]:
def hierarchical_clustering(df, n_clusters, xlabel=None, ylabel=None, method="ward"):
    cls = fcluster(linkage(df, method=method, metric="euclidean"), n_clusters, criterion="maxclust")
    plot_cluster_scatter(df, cls, n_clusters=n_clusters, start_number=1, xlabel=xlabel, title=method)
    return cls

In [ ]:
def kmeans(df, n_clusters, xlabel=None, n_init=10, random_state=10):
    cls = KMeans(n_clusters=n_clusters, n_init=n_init, random_state=random_state).fit_predict(df)
    plot_cluster_scatter(df, cls, n_clusters=n_clusters, start_number=0, xlabel=xlabel, title="{}-means".format(n_clusters))
    return cls

In [ ]:
def clustering(result, n_clusters=2, random_state=10):
    cluster = result.copy()
    for lin in ['single', 'complete', 'centroid', 'average', 'ward']:
        cls = hierarchical_clustering(result[["pred_lyric_cer"]].values, n_clusters, xlabel="pred_lyric_cer", ylabel=None, method=lin)
        cluster[lin] = cls
    cls = kmeans(result[["pred_lyric_cer"]].values, n_clusters, xlabel="pred_lyric_cer", random_state=random_state)
    cluster["kmeans"] = cls
    display(cluster)

In [ ]:
def analysis(csv_path, cluster=False):
    result = pd.read_csv(csv_path)
    display(result)
    print(result.shape)

    display(result.describe())
    display(result.corr(numeric_only=True))

    analysis_ser(result["pred_note_accuracy"], result["SV2_note_accuracy"], xlabel="predict", ylabel="SV2", title="note accuracy")
    analysis_ser(result["pred_pitch_accuracy"], result["SV2_pitch_accuracy"], xlabel="predict", ylabel="SV2", title="pitch accuracy")
    analysis_ser(result["pred_lyric_cer"], result["SV2_lyric_cer"], xlabel="predict", ylabel="SV2", title="lyric CER", reverse=True)
    if result["pred_lyric_cer"].max()>1 or result["SV2_lyric_cer"].max()>1:
        result_dropped = result.drop(index=result[result.loc[:, "pred_note_accuracy":"SV2_lyric_cer"]>1].dropna(how="all").index)
        analysis_ser(result_dropped["pred_lyric_cer"], result_dropped["SV2_lyric_cer"], xlabel="predict", ylabel="SV2", title="lyric CER (dropped)", reverse=True)

    if cluster:
        clustering(result[["title", "pred_lyric_cer"]], n_clusters=2, random_state=random_state)
        clustering(result[["title", "pred_lyric_cer"]], n_clusters=3, random_state=random_state)
        clustering(result_dropped[["title", "pred_lyric_cer"]], n_clusters=2, random_state=random_state)

#### do

In [ ]:
metafile = "meta/no7singing.csv"
resultfile = "result.csv"

execution(metafile, resultfile)
analysis(resultfile, cluster=True)